# 02. Core Product Metrics

In [ ]:
import pandas as pd
import os
import kagglehub
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

dataset_path = kagglehub.dataset_download('radistaleks/synthetic-bank-transactions')
categories    = pd.read_csv(os.path.join(dataset_path, 'categories.csv'))
clients       = pd.read_csv(os.path.join(dataset_path, 'clients.csv'))
subscriptions = pd.read_csv(os.path.join(dataset_path, 'subscriptions.csv'))
transactions  = pd.read_csv(os.path.join(dataset_path, 'transactions.csv'))

In [ ]:
clients['registration_date'] = pd.to_datetime(clients['registration_date'])
clients['birthdate']         = pd.to_datetime(clients['birthdate'])
subscriptions['date_start']  = pd.to_datetime(subscriptions['date_start'])
subscriptions['date_end']    = pd.to_datetime(subscriptions['date_end'])
transactions['date']         = pd.to_datetime(transactions['date'], format='%Y-%m-%d %H:%M:%S')

categories[categories['id'] == 28] = categories[categories['id'] == 28].fillna(4829)
categories[categories['id'] == 29] = categories[categories['id'] == 29].fillna(4900)

clients = clients.fillna(0)
subscriptions['product_company'] = subscriptions['product_company'].fillna('Неизвестно')
transactions['product_company']  = transactions['product_company'].fillna('Неизвестно')

cat_map = dict(zip(categories['id'], categories['name']))
transactions['category_name'] = transactions['product_category'].map(cat_map)
transactions['date_d'] = transactions['date'].dt.normalize()

## MAU и выручка по месяцам

In [ ]:
monthly = transactions.groupby(transactions['date'].dt.to_period('M')).agg(
    count  = ('amount', 'count'),
    total  = ('amount', 'sum'),
    avg    = ('amount', 'mean'),
    median = ('amount', 'median'),
    users  = ('client_id', 'nunique')
).sort_index()
monthly['arpu'] = monthly['total'] / monthly['users']
monthly

In [ ]:
print('Клиентов:        ', len(clients))
print('Транзакций:      ', len(transactions))
print('Выручка:         ', transactions['amount'].sum())
print('ARPU среднее:    ', monthly['arpu'].mean().round(0))
print('Средний чек:     ', transactions['amount'].mean().round(0))
print('Медианный чек:   ', transactions['amount'].median())
print('Txn/клиент/мес:  ', (monthly['count'] / monthly['users']).mean().round(1))

In [ ]:
monthly['total'].plot(figsize=(12, 4), title='Выручка по месяцам')

In [ ]:
monthly['arpu'].plot(marker='o', figsize=(12, 4), title='ARPU по месяцам')

In [ ]:
monthly['avg'].plot(marker='s', figsize=(12, 4), title='Средний чек по месяцам')

In [ ]:
monthly['count'].plot(kind='bar', figsize=(12, 4), title='Количество транзакций по месяцам')

## Категории

In [ ]:
cat_stats = transactions.groupby('category_name').agg(
    count = ('amount', 'count'),
    total = ('amount', 'sum'),
    avg   = ('amount', 'mean'),
    users = ('client_id', 'nunique')
).sort_values('total', ascending=False)
cat_stats['share'] = (cat_stats['total'] / cat_stats['total'].sum() * 100).round(2)
cat_stats

In [ ]:
cat_stats['total'].sort_values().tail(15).plot(kind='barh', figsize=(12, 8), title='Топ-15 категорий по выручке')

In [ ]:
cat_stats['count'].sort_values().tail(15).plot(kind='barh', figsize=(12, 8), title='Топ-15 категорий по кол-ву транзакций')

In [ ]:
cat_stats['avg'].sort_values().plot(kind='barh', figsize=(12, 10), title='Средний чек по категориям')

## Типы транзакций

In [ ]:
type_stats = transactions.groupby('subtype').agg(
    count  = ('amount', 'count'),
    total  = ('amount', 'sum'),
    avg    = ('amount', 'mean'),
    median = ('amount', 'median')
).sort_values('count', ascending=False)
type_stats

In [ ]:
type_stats['count'].plot(kind='bar', figsize=(10, 4), title='Кол-во транзакций по типу', rot=0)

In [ ]:
type_stats['avg'].plot(kind='bar', figsize=(10, 4), title='Средний чек по типу транзакции', rot=0)

## Временные паттерны

In [ ]:
hours = transactions.groupby(transactions['date'].dt.hour).agg(
    count = ('amount', 'count'),
    avg   = ('amount', 'mean'),
    total = ('amount', 'sum')
)
hours

In [ ]:
hours['count'].plot(figsize=(12, 4), title='Кол-во транзакций по часам', marker='o')

In [ ]:
hours['avg'].plot(figsize=(12, 4), title='Средний чек по часам', marker='o')

In [ ]:
days = transactions.groupby(transactions['date'].dt.dayofweek).agg(
    count = ('amount', 'count'),
    avg   = ('amount', 'mean'),
    total = ('amount', 'sum'),
    users = ('client_id', 'nunique')
)
days.index = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
days

In [ ]:
days['avg'].plot(kind='bar', figsize=(10, 4), title='Средний чек по дням недели', rot=0)

In [ ]:
sns.heatmap(transactions.pivot_table(
    values='amount', index=transactions['date'].dt.dayofweek,
    columns=transactions['date'].dt.hour, aggfunc='count'
), cmap='Blues', yticklabels=['Пн','Вт','Ср','Чт','Пт','Сб','Вс'])

In [ ]:
sns.heatmap(transactions.pivot_table(
    values='amount', index=transactions['date'].dt.dayofweek,
    columns=transactions['date'].dt.hour, aggfunc='mean'
), cmap='YlOrRd', yticklabels=['Пн','Вт','Ср','Чт','Пт','Сб','Вс'])

In [ ]:
# начало / середина / конец месяца
transactions.groupby(
    pd.cut(transactions['date'].dt.day, bins=[0, 10, 20, 31], labels=['начало', 'середина', 'конец']),
    observed=True
).agg(count=('amount', 'count'), avg=('amount', 'mean'), total=('amount', 'sum'))

In [ ]:
# будни vs выходные
transactions.groupby(transactions['date'].dt.dayofweek.isin([5, 6])).agg(
    count=('amount', 'count'), avg=('amount', 'mean'), median=('amount', 'median'), total=('amount', 'sum')
)

## Топ-мерчанты

In [ ]:
merchants = (
    transactions[transactions['product_company'] != 'Неизвестно']
    .groupby('product_company')
    .agg(count=('amount','count'), total=('amount','sum'), avg=('amount','mean'), users=('client_id','nunique'))
    .sort_values('total', ascending=False)
)
merchants.head(20)

In [ ]:
merchants['total'].head(15).sort_values().plot(kind='barh', figsize=(12, 7), title='Топ-15 мерчантов по выручке')

In [ ]:
merchants['users'].head(15).sort_values().plot(kind='barh', figsize=(12, 7), title='Топ-15 мерчантов по охвату')

## Проникновение продуктов

In [ ]:
N = len(clients)
active_subs = subscriptions[subscriptions['date_end'].isna()]

penetration = pd.Series({
    'Транзакции':    transactions['client_id'].nunique(),
    'Кредит':        (clients['credit'] == 1).sum(),
    'Депозит':       (clients['deposit'] == 1).sum(),
    'Подписка':      active_subs['client_id'].nunique(),
    'Музыка':        active_subs[active_subs['product_category'] == 4]['client_id'].nunique(),
    'ЖКХ/Интернет':  active_subs[active_subs['product_category'] == 29]['client_id'].nunique(),
})
(penetration / N * 100).round(1)

In [ ]:
(penetration / N * 100).plot(kind='bar', figsize=(10, 4), title=f'Проникновение продуктов, % (база {N})', rot=15)

In [ ]:
# пересечения
credit_set  = set(clients[clients['credit']  == 1]['id'])
deposit_set = set(clients[clients['deposit'] == 1]['id'])
sub_set     = set(active_subs['client_id'])

print('Кредит + Депозит:            ', len(credit_set & deposit_set))
print('Кредит + Подписка:           ', len(credit_set & sub_set))
print('Депозит + Подписка:          ', len(deposit_set & sub_set))
print('Кредит + Депозит + Подписка: ', len(credit_set & deposit_set & sub_set))